In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://www.nature.com/articles/s41467-023-36983-2#Sec34"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41590-021-00922-4/MediaObjects/41590_2021_922_MOESM2_ESM.xlsx"

# don't include in header
table_name = "41467_2023_36983_MOESM4_ESM.xlsx"
table_name = "paper_degs.xlsx" # local name # UPDATE THIS TO BE SPECIFIC FOR HILDRETH

## Read in file

In [3]:
excel = pd.read_excel(table_name, skiprows = 0)

df = excel

In [4]:
df.head()

,gene,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster
0,F13A1,0.0,3.146888,0.961,0.217,0.0,0
1,RBPJ,0.0,2.801854,0.979,0.648,0.0,0
2,FRMD4B,0.0,2.631779,0.983,0.342,0.0,0
3,PDE4D,0.0,2.389631,0.963,0.584,0.0,0
4,MAMDC2,0.0,2.343754,0.871,0.301,0.0,0


## Unfiltered

In [5]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"cluster": "cell_type_label"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["gene_id"] = None

unfiltered_df.drop(['p_val', 'avg_log2FC', 'pct.1', 'pct.2', 'p_val_adj'], axis=1, inplace=True) 
save = unfiltered_df.to_dict(orient = "records")
save

[{'gene': 'F13A1',
  'cell_type_label': 0,
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'RBPJ',
  'cell_type_label': 0,
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'FRMD4B',
  'cell_type_label': 0,
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'PDE4D',
  'cell_type_label': 0,
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'MAMDC2',
  'cell_type_label': 0,
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'MRC1',
  'cell_type_label': 0,
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'gene': 'COLEC12',
  'cell_type_label': 0,
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None

In [6]:
with open('evidence_unfiltered.json', 'w') as f:
    json.dump(save, f, indent = 4)

## Perform the filtering

In [9]:
col_pval = "p_val_adj"
col_lfc = "avg_log2FC"
col_gene = "gene"
col_ct = "cluster" # says cluster in table, but before this had "celltype"
col_rank = "pct.1" # set to None if pct. 1DNE

In [10]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [11]:
markers

,gene,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster
55598,CXCL14,3.389415e-68,2.213884,0.654,0.130,6.088067e-64,23
14258,FAM155A,0.000000e+00,2.571676,0.668,0.102,0.000000e+00,6
33333,THEMIS,0.000000e+00,2.300213,0.760,0.032,0.000000e+00,14
20942,ALCAM,0.000000e+00,2.900069,0.778,0.135,0.000000e+00,9
38076,ADGRL3,4.225859e-194,3.765187,0.810,0.095,7.590488e-190,16
...,...,...,...,...,...,...,...
44444,FABP4,7.579696e-69,2.316564,1.000,0.903,1.361465e-64,19
16444,WDPCP,6.424649e-242,2.266396,1.000,0.942,1.153996e-237,7
55594,CFD,5.273938e-72,3.443602,1.000,0.756,9.473047e-68,23
44445,B2M,2.865208e-67,2.233567,1.000,0.758,5.146487e-63,19


In [12]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster
6     0.777500
9     0.778000
20    0.842000
16    0.868400
4     0.875000
14    0.886000
21    0.892000
23    0.911917
15    0.928400
17    0.933000
2     0.934200
0     0.938375
18    0.961000
13    0.967667
3     0.969000
1     0.985333
19    0.986000
5     0.993500
8     0.998000
7     1.000000
Name: pct.1, dtype: float64

In [13]:
markers[col_ct].value_counts()

cluster
23    12
0      8
16     5
15     5
2      5
19     4
14     3
1      3
13     3
20     2
5      2
6      2
3      1
8      1
21     1
18     1
17     1
4      1
9      1
7      1
Name: count, dtype: int64

## Find Ensembl ID


In [14]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [15]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [16]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json


In [17]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = df["gene"]
df["gene_id"] = df["gene_id"].apply(run)
save = df.to_dict(orient = "records")

In [18]:
save

[{'cell_type_label': 23,
  'gene': 'CXCL14',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000145824'},
 {'cell_type_label': 6,
  'gene': 'FAM155A',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000204442'},
 {'cell_type_label': 14,
  'gene': 'THEMIS',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000172673'},
 {'cell_type_label': 9,
  'gene': 'ALCAM',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000170017'},
 {'cell_type_label': 16,
  'gene': 'ADGRL3',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000150471'},
 {'cell_type_label': 20,
  'gene': 'RAB27B',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000041353'},
 {'cell_type_label': 23,
  'gene': 'IGFBP6',
  'organi

## Save evidence

In [19]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 